In [ ]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installation completed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported successfully")

All libraries imported successfully


In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

# --- 1. Embed notes and index them in ChromaDB ---
# Load the notes from the CSV file
df = pd.read_csv('/content/college_notes.csv')

# Initialize embedding model
# This will download the model if not already cached
model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize ChromaDB client
# For simplicity, using an in-memory client for this example
client = chromadb.Client()

# Create a collection
collection_name = "college_notes_collection"
# Get or create ensures we don't error if it already exists from a previous run
collection = client.get_or_create_collection(name=collection_name)

# Prepare data for ChromaDB
# Dynamically get the name of the 4th column (index 3) to access note content
note_content_column_name = df.columns[3]
documents = df[note_content_column_name].tolist()
# Use 'topic' as the source for metadata, as a dedicated 'source' column does not exist
metadatas = [{'source': row['topic']} for index, row in df.iterrows()]
ids = [f"doc_{i}" for i in range(len(df))]

# Generate embeddings and add to ChromaDB
print("Generating embeddings and adding to ChromaDB...")
# Clear collection before adding to avoid duplicates on re-run
if collection.count() > 0:
    print(f"Clearing {collection.count()} existing documents from collection: {collection_name}.")
    client.delete_collection(collection_name)
    collection = client.get_or_create_collection(name=collection_name)

batch_size = 100
for i in range(0, len(documents), batch_size):
    batch_documents = documents[i:i+batch_size]
    batch_metadatas = metadatas[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]
    batch_embeddings = model.encode(batch_documents).tolist()
    collection.add(
        documents=batch_documents,
        embeddings=batch_embeddings,
        metadatas=batch_metadatas,
        ids=batch_ids
    )
print(f"Added {collection.count()} documents to ChromaDB collection: {collection_name}.\n")

# --- 2. Function to retrieve relevant notes ---
def retrieve_notes(query, k=3):
    # Embed the query
    query_embedding = model.encode([query]).tolist()

    # Query ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
        include=['documents', 'metadatas']
    )

    retrieved_notes = []
    if results and results['documents'] and results['metadatas']:
        for i in range(len(results['documents'][0])):
            retrieved_notes.append({
                'content': results['documents'][0][i],
                'source': results['metadatas'][0][i]['source']
            })
    return retrieved_notes

# --- 3. Setup Groq API and generate answer ---
# Set the Groq API key directly from the user's input
# In a production environment, it's recommended to use Colab secrets.
os.environ['GROQ_API_KEY'] = "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"

# Initialize Groq client
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def generate_answer_with_groq(question, context_notes):
    context_string = "\n".join([f"Source {i+1} ({note['source']}): {note['content']}" for i, note in enumerate(context_notes)])

    system_prompt = (
        "You are an AI assistant that answers questions based on the provided college notes. "
        "Provide a clear and concise answer. Cite the sources using the format '([Source X])' where X is the source number. "
        "If the answer cannot be found in the provided context, state that you don't have enough information."
    )

    user_prompt = (
        f"Question: {question}\n\n"
        f"Context:\n{context_string}\n\n"
        "Answer:"
    )

    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
        model="llama3-8b-8192", # Using a smaller, faster model. Can be changed to 'llama3-70b-8192' or 'mixtral-8x7b-32768'
        temperature=0.7,
        max_tokens=1024,
        top_p=1,
        stop=None,
        stream=False
    )
    return chat_completion.choices[0].message.content

# --- 4. Accept student question, retrieve, and generate answer ---
student_question = input("Please ask your question about the college notes: ")

# Retrieve top 3 relevant notes
top_notes = retrieve_notes(student_question, k=3)

print(f"\nRetrieved top {len(top_notes)} notes for your question: '{student_question}'")
for i, note in enumerate(top_notes):
    print(f"--- Note {i+1} (Source: {note['source']}) ---")
    print(note['content'])
    print("\n")

# Generate answer using Groq LLM
if top_notes:
    answer = generate_answer_with_groq(student_question, top_notes)
    print("\nGenerated Answer:")
    print(answer)
else:
    print("No relevant notes found to answer the question.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings and adding to ChromaDB...
Clearing 15 existing documents from collection: college_notes_collection.
Added 15 documents to ChromaDB collection: college_notes_collection.



KeyboardInterrupt: Interrupted by user